# Multi-Agent WMS Optimization (Floors 1-4 Only)

This notebook implements a detailed multi-agent optimization system for **incoming storage mode** on floors **1, 2, 3, 4** only.

Ground-floor (`0` / RDC) locations are explicitly excluded.

Implemented agents:
1. Storage Optimization Agent (weighted multi-criteria)
2. Demand Frequency Agent (ABC + exponential smoothing)
3. Slot Availability Agent (matrix + constraints)
4. Collision Avoidance Agent (reservation table)
5. Chariot Capacity Agent (greedy knapsack)
6. Coordinator Agent (orchestration pipeline)

Data source: `ai/Data/*.xlsx`

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from heapq import heappush, heappop
import math
import re

import numpy as np
import pandas as pd

DATA_DIR = Path("ai/Data")


def find_data_file(keyword: str) -> Path:
    matches = [p for p in DATA_DIR.glob("*.xlsx") if keyword.lower() in p.name.lower()]
    if not matches:
        raise FileNotFoundError(f"Missing data file containing: {keyword}")
    return matches[0]


def load_table(keyword: str) -> pd.DataFrame:
    path = find_data_file(keyword)
    df = pd.read_excel(path)
    df.columns = [str(c).strip() for c in df.columns]
    df = df.loc[:, ~df.columns.str.contains(r"^Unnamed", case=False, regex=True)]
    return df


def parse_floor_from_code(code: str) -> Optional[int]:
    text = str(code).strip().upper()

    # Format: B07-N1-XX (floors 1..4)
    m = re.match(r"^B\d{2}-N([1-4])-[A-Z0-9]+$", text)
    if m:
        return int(m.group(1))

    # Picking-like format can include leading level (0..4), e.g. 0A-01-01
    m2 = re.match(r"^([0-4])[A-Z]-\d{2}-\d{2}$", text)
    if m2:
        return int(m2.group(1))

    return None


def build_position_index(df_locations: pd.DataFrame) -> pd.DataFrame:
    df = df_locations.copy()
    df["code_emplacement"] = df["code_emplacement"].astype(str).str.strip()
    df["floor"] = df["code_emplacement"].apply(parse_floor_from_code)

    # Exclude ground floor explicitly
    df = df[df["floor"].isin([1, 2, 3, 4])].copy()

    # Generate deterministic pseudo coordinates per floor (for distance/path sim)
    df = df.reset_index(drop=True)
    floor_offsets = {1: (5, 5), 2: (45, 5), 3: (5, 45), 4: (45, 45)}

    xy: List[Tuple[int, int]] = []
    counters: Dict[int, int] = {1: 0, 2: 0, 3: 0, 4: 0}

    for _, row in df.iterrows():
        floor = int(row["floor"])
        idx = counters[floor]
        counters[floor] += 1
        ox, oy = floor_offsets[floor]
        x = ox + (idx % 30)
        y = oy + (idx // 30)
        xy.append((x, y))

    df["x"] = [x for x, _ in xy]
    df["y"] = [y for _, y in xy]

    # normalize lightweight columns
    if "type_emplacement" in df.columns:
        df["type_emplacement"] = df["type_emplacement"].astype(str).str.upper().str.strip()
    if "zone" in df.columns:
        df["zone"] = df["zone"].astype(str).str.upper().str.strip()

    return df


df_locations_raw = load_table("emplacements")
df_products_raw = load_table("produits")
df_demand_raw = load_table("historique_demande")
df_transactions_raw = load_table("transactions")
df_lines_raw = load_table("lignes_transaction")

df_locations = build_position_index(df_locations_raw)

print("Loaded tables:")
print("- locations:", len(df_locations_raw), "(usable floors 1-4:", len(df_locations), ")")
print("- products:", len(df_products_raw))
print("- demand history:", len(df_demand_raw))
print("- transactions:", len(df_transactions_raw))
print("- transaction lines:", len(df_lines_raw))

In [ ]:
@dataclass
class WarehouseSlot:
    id_emplacement: str
    code_emplacement: str
    floor: int
    zone: str
    slot_type: str
    x: int
    y: int
    is_available: bool
    max_weight: float
    reserved: bool = False
    future_allocations: int = 0

In [ ]:
class DemandFrequencyAgent:
    def __init__(self, demand_df: pd.DataFrame, alpha: float = 0.35) -> None:
        self.alpha = alpha
        self.demand_df = demand_df.copy()
        self.forecast_by_product: Dict[str, float] = {}
        self.abc_by_product: Dict[str, str] = {}
        self._fit()

    def _fit(self) -> None:
        df = self.demand_df.copy()
        if df.empty:
            return

        for col in ["id_produit", "quantite_demande"]:
            if col not in df.columns:
                return

        df["id_produit"] = df["id_produit"].astype(str).str.strip()
        df["quantite_demande"] = pd.to_numeric(df["quantite_demande"], errors="coerce").fillna(0.0)

        if "date" in df.columns:
            df["date"] = pd.to_datetime(df["date"], errors="coerce")
            df = df.sort_values(["id_produit", "date"])

        for pid, grp in df.groupby("id_produit"):
            series = grp["quantite_demande"].tolist()
            if not series:
                continue
            smoothed = series[0]
            for value in series[1:]:
                smoothed = self.alpha * float(value) + (1 - self.alpha) * smoothed
            self.forecast_by_product[pid] = float(smoothed)

        total = sum(self.forecast_by_product.values()) or 1.0
        ranked = sorted(self.forecast_by_product.items(), key=lambda kv: kv[1], reverse=True)

        cumulative = 0.0
        for pid, value in ranked:
            cumulative += value
            ratio = cumulative / total
            if ratio <= 0.8:
                self.abc_by_product[pid] = "A"
            elif ratio <= 0.95:
                self.abc_by_product[pid] = "B"
            else:
                self.abc_by_product[pid] = "C"

    def demand_score(self, product_id: str) -> float:
        pid = str(product_id)
        values = list(self.forecast_by_product.values())
        if not values:
            return 0.2
        vmin, vmax = min(values), max(values)
        val = self.forecast_by_product.get(pid, vmin)
        if math.isclose(vmin, vmax):
            return 0.5
        return (val - vmin) / (vmax - vmin)

    def abc_class(self, product_id: str) -> str:
        return self.abc_by_product.get(str(product_id), "C")

In [ ]:
class SlotAvailabilityAgent:
    def __init__(self, slots: List[WarehouseSlot]) -> None:
        self.slots = slots
        self.by_id: Dict[str, WarehouseSlot] = {s.id_emplacement: s for s in slots}

    @classmethod
    def from_dataframes(cls, locations_df: pd.DataFrame, lines_df: pd.DataFrame) -> "SlotAvailabilityAgent":
        floor_capacity = {1: 1200.0, 2: 900.0, 3: 700.0, 4: 500.0}

        occupied_ids = set()
        if not lines_df.empty and "id_emplacement_destination" in lines_df.columns:
            dst = lines_df["id_emplacement_destination"].astype(str).str.strip()
            occupied_ids = {v for v in dst if v.isdigit()}

        slots: List[WarehouseSlot] = []
        for _, row in locations_df.iterrows():
            slot_type = str(row.get("type_emplacement", "")).upper().strip()
            if slot_type not in {"PICKING", "RESERVE"}:
                continue

            sid = str(row.get("id_emplacement", "")).strip()
            if not sid.isdigit():
                continue

            floor = int(row["floor"])
            slots.append(
                WarehouseSlot(
                    id_emplacement=sid,
                    code_emplacement=str(row.get("code_emplacement", "")),
                    floor=floor,
                    zone=str(row.get("zone", "")),
                    slot_type=slot_type,
                    x=int(row.get("x", 0)),
                    y=int(row.get("y", 0)),
                    is_available=(sid not in occupied_ids),
                    max_weight=floor_capacity.get(floor, 500.0),
                )
            )

        return cls(slots)

    def hard_filter(
        self,
        sku_weight: float,
        allowed_floors: List[int],
        zone_hint: Optional[str] = None,
    ) -> List[WarehouseSlot]:
        candidates = []
        zone_hint_norm = (zone_hint or "").strip().upper()

        for slot in self.slots:
            if slot.floor not in allowed_floors:
                continue
            if not slot.is_available:
                continue
            if slot.reserved:
                continue
            if slot.max_weight < sku_weight:
                continue
            if zone_hint_norm and zone_hint_norm not in slot.zone.upper():
                continue
            candidates.append(slot)

        return candidates

    def mark_reserved(self, slot: WarehouseSlot) -> None:
        slot.reserved = True
        slot.future_allocations += 1

    def mark_occupied(self, slot: WarehouseSlot) -> None:
        slot.is_available = False
        slot.reserved = False

In [ ]:
class StorageOptimizationAgent:
    def __init__(
        self,
        demand_agent: DemandFrequencyAgent,
        dock_by_floor: Dict[int, Tuple[int, int]],
        alpha: float = 0.35,
        beta: float = 0.30,
        gamma: float = 0.20,
        delta: float = 0.15,
    ) -> None:
        self.demand_agent = demand_agent
        self.dock_by_floor = dock_by_floor
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.delta = delta

    def _distance(self, slot: WarehouseSlot) -> float:
        dx, dy = self.dock_by_floor.get(slot.floor, (0, 0))
        return abs(slot.x - dx) + abs(slot.y - dy)

    def choose_best_slot(
        self,
        product_id: str,
        sku_weight: float,
        candidates: List[WarehouseSlot],
    ) -> Tuple[Optional[WarehouseSlot], pd.DataFrame]:
        if not candidates:
            return None, pd.DataFrame()

        distances = [self._distance(s) for s in candidates]
        dmin, dmax = min(distances), max(distances)
        rows = []

        demand = self.demand_agent.demand_score(product_id)
        abc = self.demand_agent.abc_class(product_id)

        for slot, dist in zip(candidates, distances):
            if math.isclose(dmin, dmax):
                dist_norm = 0.5
            else:
                dist_norm = (dist - dmin) / (dmax - dmin)

            demand_score = demand
            distance_score = 1.0 - dist_norm
            weight_penalty = (slot.floor - 1) * max(sku_weight, 1.0) / max(slot.max_weight, 1.0)
            weight_score = max(0.0, 1.0 - weight_penalty)
            congestion_score = 1.0 / (1.0 + slot.future_allocations)

            score = (
                self.alpha * demand_score
                + self.beta * distance_score
                + self.gamma * weight_score
                + self.delta * congestion_score
            )

            rows.append(
                {
                    "id_emplacement": slot.id_emplacement,
                    "code_emplacement": slot.code_emplacement,
                    "floor": slot.floor,
                    "zone": slot.zone,
                    "abc": abc,
                    "distance": dist,
                    "demand_score": demand_score,
                    "distance_score": distance_score,
                    "weight_score": weight_score,
                    "congestion_score": congestion_score,
                    "score": score,
                }
            )

        scored = pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)
        best_id = scored.iloc[0]["id_emplacement"]
        best_slot = next(s for s in candidates if s.id_emplacement == best_id)
        return best_slot, scored

In [ ]:
class ReservationTableCollisionAgent:
    def __init__(self) -> None:
        self.reservation: Dict[Tuple[int, int, int], str] = {}

    def reserve_path(self, employee_id: str, path: List[Tuple[int, int]], start_t: int = 0) -> List[Tuple[int, int, int]]:
        result: List[Tuple[int, int, int]] = []
        t = start_t
        cur = path[0] if path else (0, 0)

        for node in path:
            nx, ny = node
            while (nx, ny, t) in self.reservation and self.reservation[(nx, ny, t)] != employee_id:
                self.reservation[(cur[0], cur[1], t)] = employee_id
                result.append((cur[0], cur[1], t))
                t += 1

            self.reservation[(nx, ny, t)] = employee_id
            result.append((nx, ny, t))
            cur = node
            t += 1

        return result

In [ ]:
class ChariotCapacityAgent:
    def __init__(self, capacity_kg: float) -> None:
        self.capacity_kg = capacity_kg

    def optimize_batch(self, items: List[Dict[str, float]]) -> Tuple[List[Dict[str, float]], float]:
        ranked = sorted(
            items,
            key=lambda x: float(x.get("priority", 0.0)) / max(float(x.get("weight", 1.0)), 1e-6),
            reverse=True,
        )

        selected: List[Dict[str, float]] = []
        total_weight = 0.0
        for item in ranked:
            w = float(item.get("weight", 0.0))
            if total_weight + w <= self.capacity_kg:
                selected.append(item)
                total_weight += w

        return selected, total_weight

In [ ]:
def manhattan_path(start: Tuple[int, int], goal: Tuple[int, int]) -> List[Tuple[int, int]]:
    x, y = start
    gx, gy = goal
    path = [(x, y)]
    while x != gx:
        x += 1 if gx > x else -1
        path.append((x, y))
    while y != gy:
        y += 1 if gy > y else -1
        path.append((x, y))
    return path

In [ ]:
class CoordinatorAgent:
    def __init__(
        self,
        demand_agent: DemandFrequencyAgent,
        slot_agent: SlotAvailabilityAgent,
        storage_agent: StorageOptimizationAgent,
        collision_agent: ReservationTableCollisionAgent,
    ) -> None:
        self.demand_agent = demand_agent
        self.slot_agent = slot_agent
        self.storage_agent = storage_agent
        self.collision_agent = collision_agent

    def process_incoming_storage(
        self,
        product_id: str,
        sku_weight: float,
        employee_id: str,
        start_coord: Tuple[int, int],
        zone_hint: Optional[str] = None,
    ) -> Dict[str, object]:
        candidates = self.slot_agent.hard_filter(
            sku_weight=sku_weight,
            allowed_floors=[1, 2, 3, 4],
            zone_hint=zone_hint,
        )

        best_slot, scored = self.storage_agent.choose_best_slot(
            product_id=product_id,
            sku_weight=sku_weight,
            candidates=candidates,
        )

        if best_slot is None:
            return {
                "status": "error",
                "message": "No valid slot for floors 1-4 under hard constraints",
                "candidates": 0,
            }

        self.slot_agent.mark_reserved(best_slot)
        route = manhattan_path(start_coord, (best_slot.x, best_slot.y))
        schedule = self.collision_agent.reserve_path(employee_id=employee_id, path=route, start_t=0)
        self.slot_agent.mark_occupied(best_slot)

        return {
            "status": "ok",
            "slot": best_slot,
            "top_scores": scored.head(10),
            "path": route,
            "time_schedule": schedule,
            "abc_class": self.demand_agent.abc_class(product_id),
        }

In [ ]:
# Build agents from project data
slot_agent = SlotAvailabilityAgent.from_dataframes(df_locations, df_lines_raw)
demand_agent = DemandFrequencyAgent(df_demand_raw)

# Expedition-like coordinates per floor for scoring distance
dock_by_floor = {
    1: (110, 10),
    2: (110, 20),
    3: (110, 50),
    4: (110, 60),
}

storage_agent = StorageOptimizationAgent(
    demand_agent=demand_agent,
    dock_by_floor=dock_by_floor,
    alpha=0.35,
    beta=0.30,
    gamma=0.20,
    delta=0.15,
 )

collision_agent = ReservationTableCollisionAgent()
coordinator = CoordinatorAgent(
    demand_agent=demand_agent,
    slot_agent=slot_agent,
    storage_agent=storage_agent,
    collision_agent=collision_agent,
)

print("Agents initialized.")
print("Available slots (floors 1-4):", sum(1 for s in slot_agent.slots if s.is_available))
print("Total slots considered (floors 1-4):", len(slot_agent.slots))

In [ ]:
# --- Demo 1: Incoming storage optimization (floors 1-4 only) ---

sample_product_id = str(df_products_raw.loc[df_products_raw["id_produit"].astype(str).str.match(r"^\d+$", na=False), "id_produit"].iloc[0])
sample_weight = 42.0

result = coordinator.process_incoming_storage(
    product_id=sample_product_id,
    sku_weight=sample_weight,
    employee_id="EMP-01",
    start_coord=(8, 8),
)

print("Storage allocation status:", result["status"])
if result["status"] == "ok":
    slot = result["slot"]
    print("Assigned slot:", slot.code_emplacement, "| floor:", slot.floor, "| zone:", slot.zone)
    print("ABC class:", result["abc_class"])
    print("Path length:", len(result["path"]))
    print("First scheduled steps:", result["time_schedule"][:8])
    display(result["top_scores"].head(5))
else:
    print(result["message"])


# --- Demo 2: Chariot capacity optimization (greedy knapsack) ---

chariot_agent = ChariotCapacityAgent(capacity_kg=320.0)
pick_items = [
    {"product_id": "31334", "weight": 80.0, "priority": 90.0},
    {"product_id": "31335", "weight": 120.0, "priority": 95.0},
    {"product_id": "31336", "weight": 50.0, "priority": 40.0},
    {"product_id": "31337", "weight": 130.0, "priority": 99.0},
    {"product_id": "31338", "weight": 70.0, "priority": 72.0},
]
selected, total_weight = chariot_agent.optimize_batch(pick_items)

print("\nChariot selection:")
print("Selected count:", len(selected), "| total weight:", total_weight)
print(pd.DataFrame(selected))